# Regression Project : House Price Prediction (Housing Dataset)
### This project focuses on building regression models to predict house prices using the housing dataset.
### The following models were implemented and evaluated:

#### Multiple Linear Regression
#### Ordinary Least Squares (OLS) Method
#### Ridge Regression
#### Lasso Regression

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

# 1. Data Cleaning Functions

In [2]:
def data_clean(dfclean):
    # list of columns to remove
    cols_to_remove = [
    "MSSubClass", "MSZoning", "Street", "LotShape",
    "LandContour", "Utilities", "LandSlope", "Neighborhood", "Condition2",
    "BldgType", "YearRemodAdd", "RoofStyle", "RoofMatl", "Exterior2nd",
    "BsmtQual", "BsmtExposure", "BsmtFinType1", "BsmtFinType2",
    "BsmtFinSF2", "BsmtUnfSF", "KitchenQual",
    "GarageFinish", "GarageArea", "GarageQual", "GarageCond",
    "MiscFeature", "MoSold"
    ]

    ### 1b. Define co-relation - Statistical evidence to elimiate the feautures
    corr_matrix = dfclean.select_dtypes(include='number').corr(method='spearman')

    
    dfclean = dfclean.drop(columns=cols_to_remove)

     #Find if any of the columns have missing values
    dfclean.isna().sum().sort_values(ascending = False)

        #Get the the columns with nulls
    df_summary = pd.DataFrame({
        "column_name": dfclean.columns,
        "null_count": dfclean.isna().sum(),
        "data_type": dfclean.dtypes
    }).reset_index(drop=True)
    df_summary

    # Handle the numerical columns null vlaues, with mean
    df_summary_num = df_summary[(df_summary["null_count"] > 0)  & (df_summary["data_type"] != 'object')]
    df_summary_num

    #Adjust NA values with mean for the numericals
    cols = df_summary_num["column_name"].tolist()
    print(cols)
    dfclean[cols] = dfclean[cols].apply(lambda x: x.fillna(x.mean()))

    #### Check for duplicate rows and remove them
    dfclean = dfclean.drop_duplicates()
        
    return dfclean

In [3]:
df_ames = pd.read_csv("C:\\Users\\vpaidmarri\\Desktop\\DS\\1_1\\ames_train.csv")
df_ames = data_clean(df_ames)
df_ames.columns.tolist()

['LotFrontage', 'MasVnrArea', 'GarageYrBlt']


['Id',
 'LotFrontage',
 'LotArea',
 'Alley',
 'LotConfig',
 'Condition1',
 'HouseStyle',
 'OverallQual',
 'OverallCond',
 'YearBuilt',
 'Exterior1st',
 'MasVnrType',
 'MasVnrArea',
 'ExterQual',
 'ExterCond',
 'Foundation',
 'BsmtCond',
 'BsmtFinSF1',
 'TotalBsmtSF',
 'Heating',
 'HeatingQC',
 'CentralAir',
 'Electrical',
 '1stFlrSF',
 '2ndFlrSF',
 'LowQualFinSF',
 'GrLivArea',
 'BsmtFullBath',
 'BsmtHalfBath',
 'FullBath',
 'HalfBath',
 'BedroomAbvGr',
 'KitchenAbvGr',
 'TotRmsAbvGrd',
 'Functional',
 'Fireplaces',
 'FireplaceQu',
 'GarageType',
 'GarageYrBlt',
 'GarageCars',
 'PavedDrive',
 'WoodDeckSF',
 'OpenPorchSF',
 'EnclosedPorch',
 '3SsnPorch',
 'ScreenPorch',
 'PoolArea',
 'PoolQC',
 'Fence',
 'MiscVal',
 'YrSold',
 'SaleType',
 'SaleCondition',
 'SalePrice']

# 2. Data scaling and Pre-processing

#### Features scaling


In [4]:
def data_scale(dfscale):
    
    #### Features - Scalar fit, feature with different ranges 
    # Columns to scale
    cols_to_scale = ['LotFrontage', 'LotArea']
    
    # Initialize scaler
    scaler = StandardScaler()
    
    # Fit and transform all columns at once
    dfscale[cols_to_scale] = scaler.fit_transform(dfscale[cols_to_scale])

    #### Features - Robust fit, feature with different ranges and outliers
     # Initialize scaler
    scaler = RobustScaler()
    # Columns to scale that eligible for robust
    cols_to_scale = ['MasVnrArea','BsmtFinSF1','TotalBsmtSF','1stFlrSF','2ndFlrSF','LowQualFinSF','GrLivArea','WoodDeckSF','OpenPorchSF','EnclosedPorch','3SsnPorch','PoolArea']

    # Fit and transform all columns at once
    dfscale[cols_to_scale] = scaler.fit_transform(dfscale[cols_to_scale])

    # variable PoolArea convert to a binary
    # Only few houses has the Pool, scaling would effect the actual values hence add a new binary column 
    dfscale["HasPool"] = (dfscale['PoolArea'] > 0).astype(int)
    #df_ames.drop(columns='PoolArea')
    dfscale[dfscale["HasPool"] == 1].head()

    return dfscale
   

In [5]:
df_ames = pd.read_csv("C:\\Users\\vpaidmarri\\Desktop\\DS\\1_1\\ames_train.csv")
df_ames = data_clean(df_ames)
data_scale(df_ames)

['LotFrontage', 'MasVnrArea', 'GarageYrBlt']


,Id,LotFrontage,LotArea,Alley,LotConfig,Condition1,HouseStyle,OverallQual,OverallCond,YearBuilt,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscVal,YrSold,SaleType,SaleCondition,SalePrice,HasPool
0,1,-0.229372,-0.207142,NaN,Inside,Norm,2Story,7,5,2003,...,0,0.0,NaN,NaN,0,2008,WD,Normal,208500,0
1,2,0.451936,-0.091886,NaN,FR2,Feedr,1Story,6,8,1976,...,0,0.0,NaN,NaN,0,2007,WD,Normal,181500,0
2,3,-0.093110,0.073480,NaN,Inside,Norm,2Story,7,5,2001,...,0,0.0,NaN,NaN,0,2008,WD,Normal,223500,0
3,4,-0.456474,-0.096897,NaN,Corner,Norm,2Story,7,5,1915,...,0,0.0,NaN,NaN,0,2006,WD,Abnorml,140000,0
4,5,0.633618,0.375148,NaN,FR2,Norm,2Story,8,5,2000,...,0,0.0,NaN,NaN,0,2008,WD,Normal,250000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1455,1456,-0.365633,-0.260560,NaN,Inside,Norm,2Story,6,5,1999,...,0,0.0,NaN,NaN,0,2007,WD,Normal,175000,0
1456,1457,0.679039,0.266407,NaN,Inside,Norm,1Story,6,6,1978,...,0,0.0,NaN,MnPrv,0,2010,WD,Normal,210000,0
1457,1458,-0.183951,-0.147810,NaN,Inside,Norm,2Story,7,9,1941,...,0,0.0,NaN,GdPrv,2500,2010,WD,Normal,266500,0
1458,1459,-0.093110,-0.080160,NaN,Inside,Norm,1Story,5,6,1950,...,0,0.0,NaN,NaN,0,2010,WD,Normal,142125,0


# Handle the Categorical columns null vlaues

In [6]:
def data_categorical_columns(df_category):
    #Get the the columns with nulls
    df_summary = pd.DataFrame({
        "column_name": df_category.columns,
        "null_count": df_category.isna().sum(),
        "data_type": df_category.dtypes
    }).reset_index(drop=True)

    # Categorical columns
    df_summary_cat = df_summary[(df_summary["null_count"] > 0)  & (df_summary["data_type"] == 'object')]

    cols1 = df_summary_cat["column_name"].tolist()
    #cols1
    
    # Remove the columns with more null values more than 40 percentage
    missing_pct = df_category[cols1].isnull().mean() * 100
    cols_to_drop = missing_pct[missing_pct > 40].index
    #print('cols_to_drop', cols_to_drop)
    df_category = df_category.drop(columns=cols_to_drop)
    
    # Keep only columns that exist in df_category
    available_cols = [col for col in cols1 if col in df_category.columns]

    # Replace with Missing for NA
    for col in available_cols:
        df_category[col] = df_category[col].fillna('Missing')

    # Onehot encoding
    encoder = OneHotEncoder(drop='first',sparse_output=False)
    encoded = encoder.fit_transform(df_category[available_cols])
    # Convert to DataFrame
    encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out())
    df_ames_Final=pd.concat([df_category, encoded_df], axis=1)
    return df_category

#### implement co-relation and eliminate the respective columns

In [7]:
def data_preprocessing(df_corr):
    corr_matrix = df_corr.select_dtypes(include='number').corr(method='spearman')
    # Set up the matplotlib figure
    #plt.figure(figsize=(20, 20))    
    # Generate heatmap
    #sns.heatmap(
     #   corr_matrix,
      #  annot=True,        # Show correlation values
       # cmap='coolwarm',   # Color map
        #fmt=".2f",         # Format for numbers
        #linewidths=0.5
    #)
    
    # Add title
    #plt.title("Correlation Heatmap")    
    # Show plot
    #plt.tight_layout()
    #plt.show()

    # Select the final list of features target > 0.4
    target_corr = corr_matrix['SalePrice'].abs()

    # Select features with correlation > 0.4
    selected_features = target_corr[target_corr > 0.4].index

    #Final set of dataset
    df_corr = df_corr[selected_features]

    return df_corr

# 3. Data Modelling
#### Linear regression - Multiple features

In [8]:
def model_linearregression(df_train, df_test):
    df_train = df_train.copy()
    df_test = df_test.copy()
    X = df_train.drop(columns=['SalePrice'])
    y = df_train['SalePrice']
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=0
    )
    
    model = LinearRegression().fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_pred
    
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)

    print("*****************Multiple Linear Regression**************************")
    print("R2:", r2_score(y_test, y_pred))
    print("MSE:", mse)
    print("RMSE:", rmse) 
    
    X_test = df_test[X.columns.tolist()]
    
    y_pred = model.predict(X_test)
    print(y_pred)

#### Log Transformation

In [9]:
def model_linear_reg_logtrasformation(df_train, df_test):
    df_train = df_train.copy()
    df_test = df_test.copy()
    df_train['SalePrice_log'] = np.log(df_train['SalePrice'])

    X = df_train.drop(columns=['SalePrice','SalePrice_log'])
    y = df_train['SalePrice_log']
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=0
    )
    
    model = LinearRegression().fit(X_train, y_train)
    y_pred_log = model.predict(X_test)
    
    y_pred_actual = np.exp(y_pred_log)
    y_test_actual = np.exp(y_test)
    
    mse = mean_squared_error(y_test, y_pred_log)
    print("mse actual:,", mean_squared_error(y_test_actual, y_pred_actual))
    
    rmse = np.sqrt(mse)

    print("*****************Log Transformation - Multiple Linear Regression**************************")
    print("Train_R2:", r2_score(y_test, y_pred_log))
    print("Train MSE:", mse)
    print("Train RMSE:", rmse)

    X_test = df_test[X.columns.tolist()]
    
    y_pred_log = model.predict(X_test)
    y_pred_actual = np.exp(y_pred_log)
    print(y_pred_log)
    print(y_pred_actual)

#### OLS Model

In [10]:
# Importing the stats library
import statsmodels.api as sm

def model_OLS(df_train, df_test):
    df_train = df_train.copy()
    df_test = df_test.copy()
    X = df_train.drop(columns=['SalePrice'])
    y = df_train['SalePrice']
    X = sm.add_constant(X) # adding a constant
    #X_train, X_test, y_train, y_test 
    #lr_model = sm.OLS(y,X).fit()
    lr_model = sm.OLS(y,X).fit()

    X_test = df_test[df_train.drop(columns=['SalePrice']).columns.tolist()]
    X_test = sm.add_constant(X_test) 
    predictions = lr_model.predict(X_test)  
    print_model = lr_model.summary()
    print(print_model)
    
    p_values = lr_model.pvalues
    significant_columns = p_values[p_values < 0.05 ].index.tolist()
    print(significant_columns)

#### Ridge Regression Model

In [11]:
from sklearn.linear_model import Ridge

def model_Ridge(df_train, df_test, alpha_score=0.3):
    df_train = df_train.copy()
    df_test = df_test.copy()
    print("*****************Ridge Regression**************************")

    X = df_train.drop(columns=['SalePrice'])
    y = df_train['SalePrice']
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=0
    )

    print(f"alpha_score:{alpha_score}")
    l_ridge = Ridge(alpha=alpha_score, random_state=0)
    l_ridge.fit(X_train, y_train)

  
    ridge_predict = l_ridge.predict(X_test)

    
    # Calculating MSE    
    ridge_mse = np.mean((ridge_predict - y_test)** 2)
    rmse = np.sqrt(ridge_mse)
    print("Mean Squared Error",ridge_mse)
    print("RMSE",rmse)
    
    # Accuracy Score
    score = l_ridge.score(X_test,y_test)
    print("Accuracy",score)

    X_test = df_test[X.columns.tolist()]
    ridge_predict = l_ridge.predict(X_test)
    print(ridge_predict)

#### Lasso Regression Model

In [12]:
from sklearn.linear_model import Lasso

def model_Lasso(df_train, df_test, alpha_score=0.3):
    df_train = df_train.copy()
    df_test = df_test.copy()
    
    print("*****************Lasso Regression**************************")

    X = df_train.drop(columns=['SalePrice'])
    y = df_train['SalePrice']
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=0
    )

    print(f"alpha_score:{alpha_score}")
    l_lasso = Lasso(alpha=alpha_score, random_state=0)
    l_lasso.fit(X_train, y_train)
    lasso_predict = l_lasso.predict(X_test)
    
    # Calculating MSE    
    lasso_mse = np.mean((lasso_predict - y_test)** 2)
    rmse = np.sqrt(lasso_mse)
    print("Mean Squared Error",lasso_mse)
    print("RMSE",rmse)
    
    # Accuracy Score
    score = l_lasso.score(X_test,y_test)
    print("Accuracy",score)

    X_test = df_test[X.columns.tolist()]
    lasso_predict = l_lasso.predict(X_test)
    print(lasso_predict)

## Create a pipeline to process the dataset
#### Train dataset

In [13]:
prep_pipeline = Pipeline([
    ('clean', FunctionTransformer(data_clean)),
    ('scale', FunctionTransformer(data_scale)),
    ('categorical', FunctionTransformer(data_categorical_columns)),
    ('preprocess', FunctionTransformer(data_preprocessing))
])
df_ames_train = pd.read_csv("C:\\Users\\vpaidmarri\\Desktop\\DS\\1_1\\ames_train.csv")
df_ames_train = prep_pipeline.fit_transform(df_ames_train)
df_ames_train.head()

['LotFrontage', 'MasVnrArea', 'GarageYrBlt']


,LotArea,OverallQual,YearBuilt,MasVnrArea,TotalBsmtSF,1stFlrSF,GrLivArea,FullBath,TotRmsAbvGrd,Fireplaces,GarageYrBlt,GarageCars,OpenPorchSF,SalePrice
0,-0.207142,7,2003,1.193303,-0.269652,-0.453608,0.380070,2,8,0,2003.0,2,0.529412,208500
1,-0.091886,6,1976,0.000000,0.538308,0.343643,-0.312090,2,6,1,1976.0,2,-0.367647,181500
2,0.073480,7,2001,0.986301,-0.142289,-0.327933,0.497489,2,6,1,2001.0,2,0.250000,223500
3,-0.096897,7,1915,0.000000,-0.468657,-0.247423,0.390885,1,7,1,1998.0,3,0.147059,140000
4,0.375148,8,2000,2.130898,0.305473,0.113893,1.134029,2,9,1,2000.0,3,0.867647,250000


In [14]:

prep_pipeline = Pipeline([
    ('clean', FunctionTransformer(data_clean)),
    ('scale', FunctionTransformer(data_scale)),
    ('categorical', FunctionTransformer(data_categorical_columns))
])

df_ames_test = pd.read_csv("C:\\Users\\vpaidmarri\\Desktop\\DS\\1_1\\ames_test.csv")
df_ames_test = prep_pipeline.fit_transform(df_ames_test)
df_ames_test.head()

['LotFrontage', 'MasVnrArea', 'BsmtFinSF1', 'TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath', 'GarageYrBlt', 'GarageCars']


,Id,LotFrontage,LotArea,LotConfig,Condition1,HouseStyle,OverallQual,OverallCond,YearBuilt,Exterior1st,...,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,YrSold,SaleType,SaleCondition,HasPool
0,1461,0.555587,0.363929,Inside,Feedr,1Story,5,6,1961,VinylSd,...,-0.388889,0.0,0.0,120,0.0,0,2010,WD,Normal,0
1,1462,0.604239,0.897861,Corner,Norm,1Story,6,6,1958,Wd Sdng,...,0.111111,0.0,0.0,0,0.0,12500,2010,WD,Normal,0
2,1463,0.263676,0.809646,Inside,Norm,2Story,5,5,1997,VinylSd,...,0.083333,0.0,0.0,0,0.0,0,2010,WD,Normal,0
3,1464,0.458284,0.032064,Inside,Norm,2Story,6,6,1998,VinylSd,...,0.111111,0.0,0.0,0,0.0,0,2010,WD,Normal,0
4,1465,-1.244533,-0.971808,Inside,Norm,1Story,8,5,1992,HdBoard,...,0.750000,0.0,0.0,144,0.0,0,2010,WD,Normal,0


In [15]:
model_linearregression(df_ames_train,df_ames_test)
model_OLS(df_ames_train,df_ames_test)
model_linear_reg_logtrasformation(df_ames_train,df_ames_test)
print("After log transformation", df_ames_train.info())
model_Ridge(df_ames_train,df_ames_test,3)
model_Lasso(df_ames_train,df_ames_test,50)


*****************Multiple Linear Regression**************************
R2: 0.6542005874106732
MSE: 2388038860.982456
RMSE: 48867.564508398165
[106435.74457367 169890.83698025 176781.8002856  ... 163088.70871386
 111399.25475563 241887.59778391]
                            OLS Regression Results                            
Dep. Variable:              SalePrice   R-squared:                       0.782
Model:                            OLS   Adj. R-squared:                  0.780
Method:                 Least Squares   F-statistic:                     398.9
Date:                Tue, 17 Mar 2026   Prob (F-statistic):               0.00
Time:                        23:01:06   Log-Likelihood:                -17432.
No. Observations:                1460   AIC:                         3.489e+04
Df Residuals:                    1446   BIC:                         3.497e+04
Df Model:                          13                                         
Covariance Type:            nonrobust        